In [74]:
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from create_dataset_utils import *
from sklearn.model_selection import StratifiedKFold
from scipy.stats import pearsonr
from tableone import TableOne
from statsmodels.stats.proportion import proportion_confint
import json
from CDR_assessment_utils import *
import warnings
from matplotlib.patches import Rectangle
from scipy import stats
import tableone
warnings.filterwarnings('ignore')
import statsmodels.api as sm
import nibabel as nib
from sklearn.metrics import confusion_matrix

In [3]:
TICKS_FONTSIZE = 12
LABEL_FONTSIZE = 16
TITLE_FONTSIZE = 16
MATCHING_BINS = 6 # maximum 5
MATCHING_BINS2 = 3

FINAL_PRED_COL = "final_pred"
BEST_FINAL_PRED_COL = "best_final_pred"
IQ_DATASET_COL = "IQ_dataset"
RAD_DESC_COL = "rad_description"
RAD_SCORE_COL = "rad_score"
RAD_OVER_PRED_COL = "rad_over_pred"
USE_PT_LEVEL_ANALYSIS_COL = "use_pt_level"
dataset_dict_key_list = ["T2_v3"]
EXCLUSE_ALREADY_IDENTIFIED_THR_UROLIFT = True

CDR_FIGURE_TABLE_PATH = Path("/research/projects/Nakai/clf_artifacts/CDR_Figure_Table_onlyDL")

In [30]:
# read patient-level dataset containing best IQ, age, PSA, facility, pre-MRI biopsy status 
acc_level_df = pd.read_excel("T2_acclevel.xlsx")

# IQ categorization for CDR analysis using FINAL_PRED_COL
pT2_0, pT2_1, pT2_2, pT2_3 = [acc_level_df[acc_level_df[FINAL_PRED_COL]==x] for x in range(4)]

# Target group (degraded IQ with score of 0 to 2)
pT2_target_df_list = [pT2_0, pT2_1, pT2_2]

# Control group with the best IQ of 3
list_of_control_pool = [pT2_3]

list_of_target_df_list = [pT2_target_df_list]
score_list = ["0", "1", "2"]

In [31]:
matching_col = [f"age_int_bins{MATCHING_BINS}", f"PSA_bins{MATCHING_BINS}", "facility", "Pre_MR_Dx_from_MRI_MedTagger"]

if MATCHING_BINS2:
    matching_col2 = [f"age_int_bins{MATCHING_BINS2}", f"PSA_bins{MATCHING_BINS2}", "facility", "Pre_MR_Dx_from_MRI_MedTagger"]

def matching(target_df, pool_df, matching_col, random_state=0):
    target_df = target_df.reset_index(drop=True)
    pool_df = pool_df.reset_index(drop=True)
    
    unmatched_target_list = []
    matched_case_list = ["sample"]
    sample_n = 20
    matching_fail = False
    print("Matching start..")
    max_match_n = 0
    for match_n in range(sample_n):
        sample_list = []
        for j, row in target_df.iterrows():
            if matching_fail==False:
                cand = pool_df[(pool_df[matching_col] == row[matching_col]).all(axis=1)]
                if MATCHING_BINS2:
                    cand2 = pool_df[(pool_df[matching_col2] == row[matching_col2]).all(axis=1)]
                # if candidate exists
                if len(cand)>=1:
                    matched_case = cand.sample(1, random_state=random_state)
                    sample_list.append(matched_case)
                    matched_case_acc = matched_case["acc"].values[0]
                    pool_df = pool_df[pool_df["acc"]!=matched_case_acc] # updated the pool by removing the acc
                    if match_n!=0:
                        if (j==(len(target_df)-1)):
                            matched_case_list.append(sample_list)
                    else:
                        matched_case_list[match_n] = sample_list
                else:
                    if len(cand2)>=1:
                        matched_case = cand2.sample(1, random_state=random_state)
                        sample_list.append(matched_case)
                        matched_case_acc = matched_case["acc"].values[0]
                        pool_df = pool_df[pool_df["acc"]!=matched_case_acc] # updated the pool by removing the acc
                        if match_n!=0:
                            if (j==(len(target_df)-1)):
                                matched_case_list.append(sample_list)
                        else:
                            matched_case_list[match_n] = sample_list
                    else:
                        if match_n>=1:
                            print(f"-Successful 1:{match_n} matching")
                            matching_fail=True
                            if max_match_n==0:
                                max_match_n = match_n
                        else:
                            unmatched_target_list.append(row)
            else:
                break
    if len(unmatched_target_list)>=1:
        print(f"Unmatched target: {len(unmatched_target_list)} exams")
    matched_control = pd.concat([g for c in matched_case_list for g in c])
    return matched_control, unmatched_target_list

In [32]:
# Matching. Store populations of target group (score 0-2), control group (score 3), and matched control group in dictionaries
target_dict = {}
target_dict_before_matching = {}
control_dict = {}
matched_control_dict = {}
for target_df_list, pool_df, category in zip(list_of_target_df_list,
                                             list_of_control_pool,
                                             name_list):
    for target_df, score in zip(target_df_list, score_list):
        key = f"{category}_{score}"
        print(key)
        matched_control, unmatched_target_list = matching(target_df, pool_df, matching_col, random_state=1)
        matched_control_dict[key] = matched_control
        if len(unmatched_target_list)>=1:
            target_df = target_df.reset_index(drop=True)
            target_df_after_exc_unmatch = target_df.drop(pd.concat([pd.DataFrame(x).T for x in unmatched_target_list]).index)
            print(f"target_df:{len(target_df)}")
            print(f"target_df_after_exc_unmatch:{len(target_df_after_exc_unmatch)}")
            target_dict[key] = target_df_after_exc_unmatch
        else:
            target_dict[key] = target_df
        target_dict_before_matching[key] = target_df
        control_dict[key] = pool_df

T2_0
Matching start..
-Successful 1:4 matching
T2_1
Matching start..
-Successful 1:1 matching
Unmatched target: 48 exams
target_df:1447
target_df_after_exc_unmatch:1399
T2_2
Matching start..
-Successful 1:1 matching
Unmatched target: 3227 exams
target_df:6602
target_df_after_exc_unmatch:3375


In [38]:
# Point plots (CDR, AIR, PPV, Bx rates)
INCLUDE_SCORE01_IN_POINTPLOT = False
for category in name_list:
    dataset_dict_key = f"{category}_v3"
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8,8))
    ax = axes.flatten()
    for i, metric in enumerate(["CDR", "AIR", "PPV", "Biopsy rate"]):
        #category = dataset_dict_key.strip("_v3")
        _, _, plt_title, _, score_description, _, _, _ = settings(dataset_dict, dataset_dict_key)
        
        xlim_margin = 0.4
        score_list_ = score_list
        xlim_min, xlim_max = -xlim_margin, 2+xlim_margin
        
        target_metric = [concat_stats_df.loc[metric][category, score] for score in score_list_]
        target_metric_ci = [concat_stats_df.loc[f"{metric}_CIs"][category, score] for score in score_list_]
        target_metric_l, target_metric_h = [x[0] for x in target_metric_ci], [x[1] for x in target_metric_ci]
        target_metric_text_list = [f"{m:.2f}\n({l:.2f}-{h:.2f})" for m, l, h in zip(target_metric, target_metric_l, target_metric_h)]
        
        target_df_for_plot = pd.DataFrame({"score":score_list_, category:["Target"]*len(score_list_), "m":target_metric, "l":target_metric_l, "h":target_metric_h})
        
        mc_metric = [concat_stats_df.loc[metric][category, f"mc_{score}"] for score in score_list_]
        mc_metric_ci = [concat_stats_df.loc[f"{metric}_CIs"][category, f"mc_{score}"] for score in score_list_]
        mc_metric_l, mc_metric_h = [x[0] for x in mc_metric_ci], [x[1] for x in mc_metric_ci]
        mc_metric_text_list = [f"{m:.2f}\n({l:.2f}-{h:.2f})" for m, l, h in zip(mc_metric, mc_metric_l, mc_metric_h)]

        mc_df_for_plot = pd.DataFrame({"score":score_list_, category:["Matched control"]*len(score_list_), "m":mc_metric, "l":mc_metric_l, "h":mc_metric_h})
        
        concat_df_for_plot = pd.concat([target_df_for_plot, mc_df_for_plot])

        # Pointplots
        #plt.figure(figsize=(figwidth,4))
        sns.pointplot(data = concat_df_for_plot, x="score", y="m", dodge=True, hue=category, palette=[sns.color_palette("colorblind")[x] for x in [1,0]], ax=ax[i])
        
        ylim = {"CDR":0.37, "AIR":0.6, "PPV":1, "Biopsy rate":1}[metric]
        yticks = {"CDR":[0, 0.1, 0.2, 0.3], "AIR":[0, 0.2, 0.4, 0.6], 
                  "PPV":[0, 0.2, 0.4, 0.6, 0.8, 1.0], "Biopsy rate":[0, 0.2, 0.4, 0.6, 0.8, 1.0]}[metric]
        ax[i].set_ylim(0,ylim)
        ax[i].set_yticks(yticks, yticks, fontsize=TICKS_FONTSIZE*1.2)
        
        # CIs vertical lines
        positioning_x = 0.025
        for color_index, df_ in enumerate([target_df_for_plot, mc_df_for_plot]):
            #display(df_)
            df_.reset_index(drop=True, inplace=True)
            for x, ymin, ymax in zip(df_.index, df_["l"], df_[f"h"]):
                x_plot = (x+(2*color_index-1)*positioning_x)
                ymin_plot = ymin/ylim
                ymax_plot = ymax/ylim
                #print(x_plot, ymin, ymax)
                ax[i].axvline(x_plot, ymin_plot, ymax_plot, lw=2, color=sns.color_palette("colorblind")[1-color_index])

        # add text (calculated value + 95%CI)
        PPV_y = 0.28
        ylim_prop = 0.115
        for j, target_val, mc_val in zip(range(4), target_metric_text_list, mc_metric_text_list):
            y_target = {"T2_v3":{"CDR":PPV_y*ylim, "AIR":PPV_y*ylim, "PPV":PPV_y, "Biopsy rate":PPV_y},
                        "DWI_metal_v3":{"CDR":PPV_y*ylim, "AIR":PPV_y*ylim, "PPV":PPV_y, "Biopsy rate":PPV_y},
                        "DWI_gas_v3":{"CDR":PPV_y*ylim, "AIR":PPV_y*ylim, "PPV":PPV_y, "Biopsy rate":PPV_y}}[dataset_dict_key][metric]
            y_mc = y_target - (ylim*ylim_prop)
            ax[i].text(j,y_target, target_val, ha="center", va="center", c=sns.color_palette("colorblind")[1]) # details_T[metric][j]-0.2  
            ax[i].text(j,y_mc, mc_val, ha="center", va="center", c=sns.color_palette("colorblind")[0])

        ax[i].set_xticks([0,1,2,3], (score_description[::-1][:-1] + ["Poor-\nNondiagnostic"]), fontsize=TICKS_FONTSIZE*1.2, ha="center")
        #ax[i].set_xlabel("Image quality score", fontsize=LABEL_FONTSIZE)
        #ax[i].set_ylabel("")
        ax[i].set(xlabel=None, ylabel=None)
        ax[i].set_xlim(xlim_min, xlim_max)
        ax[i].set_title(f"{metric}", fontsize=LABEL_FONTSIZE)
        L=ax[i].legend(loc="lower center",ncols=2, fontsize=9)
        L.get_texts()[1].set_text(f'Matched control ({score_description[0]})')
        #else:
        #    ax[i].legend().set_visible(False)
    #plt.suptitle(plt_title, fontsize=TITLE_FONTSIZE)
    plt.tight_layout()
    figure_path = Path.joinpath(CDR_FIGURE_TABLE_PATH, f"{category}.jpeg")
    plt.savefig(figure_path, dpi=600)
    plt.clf()

<Figure size 576x576 with 0 Axes>

In [39]:
# Risk ratios 
def extract_risk_ratio(category, score, metric, concat_stats_df=concat_stats_df):
    metric_numerator = {"CDR":"cancer_detected_exams", "AIR":"abnormal_exams", "PPV":"cancer_detected_exams", "Biopsy rate":"abnormal_exams_with_path"}[metric]
    metric_denominator = {"CDR":"total_num_exams", "AIR":"total_num_exams", "PPV":"abnormal_exams_with_path", "Biopsy rate":"abnormal_exams"}[metric]
    
    def extract_value_from_concat_stats_df(category, score, metric, concat_stats_df=concat_stats_df):
        target_metric = concat_stats_df.loc[metric][category][score]
        mc_metric = concat_stats_df.loc[metric][category][f"mc_{score}"]
        return target_metric, mc_metric

    # numerator (positive case)
    target_metric_pos, mc_metric_pos = extract_value_from_concat_stats_df(category, score, metric_numerator, concat_stats_df=concat_stats_df)
    
    # denominator (positive and negative case)
    target_metric_pos_neg, mc_metric_pos_neg = extract_value_from_concat_stats_df(category, score, metric_denominator, concat_stats_df=concat_stats_df)
    
    # negative case
    target_metric_neg = target_metric_pos_neg - target_metric_pos
    mc_metric_neg = mc_metric_pos_neg - mc_metric_pos

    # https://www.statsmodels.org/dev/generated/statsmodels.stats.contingency_tables.Table2x2.html
    # The two rows define population subgroups, column 0 is the number of ‘events’, and column 1 is the number of ‘non-events’.
    metric_table2x2 = np.array([[target_metric_pos, target_metric_neg], [mc_metric_pos, mc_metric_neg]])
    metric_table2x2_sm = sm.stats.Table2x2(metric_table2x2)
    RR = metric_table2x2_sm.riskratio
    RR_l = metric_table2x2_sm.riskratio_confint()[0]
    RR_u = metric_table2x2_sm.riskratio_confint()[1]
    RR_pvalue = metric_table2x2_sm.riskratio_pvalue()
    return RR, RR_l, RR_u, RR_pvalue

In [40]:
# Point plots of risk ratios (CDR, AIR, PPV, Bx rates)

list_for_x = score_list_ # or score_list_, Used for x-axis in the ratio plot

for category in name_list:
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8,8))
    ax = axes.flatten()
    color = "dimgray" #sns.color_palette("colorblind")[1]
    text_color = "black"
    ylim = 1.7
    fontsize_prop = {2:1.2, 3:1.2, 4:1}[len(list_for_x)]
    
    for i, metric in enumerate(["CDR", "AIR", "PPV", "Biopsy rate"]):
        for x, score in enumerate(list_for_x): # or score_list_
            dataset_dict_key = f"{category}_v3"
            _, _, plt_title, _, score_description, _, _, _ = settings(dataset_dict, dataset_dict_key)
            RR, RR_l, RR_u, RR_pvalue = extract_risk_ratio(category, score, metric, concat_stats_df=concat_stats_df)
            RR_text = f"{RR:.2f}\n({RR_l:.2f}-{RR_u:.2f})"
            ax[i].scatter(x, RR, color=color)
            ax[i].hlines(1, xlim_min, xlim_max, lw=1, color="gray", linestyle=":")
            ax[i].vlines(x, RR_l, RR_u, lw=2, color=color)
            
            # metrics value
            ax[i].text(x, 0.5, RR_text, ha="center", va="center", c=text_color, fontsize=9.5*fontsize_prop)
            
            # p value
            if metric=="CDR":
                if RR_pvalue>0.005:
                    P = f"{RR_pvalue:.2f}"
                elif RR_pvalue<0.001:
                    P = "<0.001"
                else:
                    P = f"{RR_pvalue:.3f}"
                ax[i].text(x, 0.25, f"P={P}", ha="center", va="center", c=text_color, fontsize=9.5*fontsize_prop, bbox=dict(facecolor="none", edgecolor="gray"))
            
            xlim_margin = 0.42
            xlim_min, xlim_max = -xlim_margin, len(list_for_x)-1+xlim_margin
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(0, ylim)
            #ax[i].set_xticks(xlim_min, xlim_max)
            ax[i].set_yticks([0, 0.33, 0.66, 1.00, 1.33, 1.66], [0, 0.33, 0.66, 1.00, 1.33, 1.66],  fontsize=TICKS_FONTSIZE)
            
            xticks = score_description[::-1][:3]
            if len(list_for_x)==4:
                xticks+=[f"{score_description[-1]}\n-{score_description[-2]}"]
            ax[i].set_xticks(range(len(list_for_x)), xticks,  fontsize=TICKS_FONTSIZE*fontsize_prop)
            ax[i].set_title(f"{metric} ratio", fontsize=LABEL_FONTSIZE)
    plt.tight_layout()
    
    figure_path = Path.joinpath(CDR_FIGURE_TABLE_PATH, f"{category}_ratio.jpeg")
    plt.savefig(figure_path, dpi=600)
    plt.clf()

<Figure size 576x576 with 0 Axes>